# ChEMBL GLP1 Exploratory Data Analysis

Data obtained from ChEMBL filtering for GLP1 target

#### Setup libraries and wide pandas view:

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/raw/chembl_glp1.csv", sep=';')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

C:\Users\Tom\AppData\Local\Temp\ipykernel_1536\361401215.py:6: DtypeWarning: Columns (0: Assay Subcellular Fraction) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/chembl_glp1.csv", sep=';')


* Pandas raises a DtypeWarning on 'Assay Subcellular Fraction' due to chunked type inference (NaN-only chunks read  as float, but the value-containing chunks as objects). Not a genuine mixed type problem, all the values are strings as found below.

## Do AlogP or Molecular Weight predict GLP-1 potency in this ChEMBL dataset?

### *Part 1 - Structural Audit*

#### Datatypes


In [2]:
print(df.head(5), '\n')
print(df.info(), '\n')

  Molecule ChEMBL ID Molecule Name  Molecule Max Phase  Molecular Weight  #RO5 Violations  AlogP  Compound Key                                             Smiles Standard Type Standard Relation  Standard Value Standard Units  pChEMBL Value Data Validity Comment                Comment    Uo Units  Ligand Efficiency BEI  Ligand Efficiency LE  Ligand Efficiency LLE  Ligand Efficiency SEI  Potential Duplicate Assay ChEMBL ID                                  Assay Description Assay Type BAO Format ID          BAO Label Assay Organism  Assay Tissue ChEMBL ID  Assay Tissue Name Assay Cell Type Assay Subcellular Fraction  Assay Parameters  Assay Variant Accession  Assay Variant Mutation Target ChEMBL ID                       Target Name Target Organism     Target Type Document ChEMBL ID  Source ID     Source Description Document Journal  Document Year Cell ChEMBL ID Properties Action Type Standard Text Value   Value
0      CHEMBL4095693           NaN                 NaN            518.80      

* All datatypes appear to be correct. float64(17), int64(2), str(29)

In [6]:
type_counts = df.apply(lambda col: col.dropna().map(type).nunique())
print('Top five column type counts:', '\n', type_counts.sort_values(ascending=False).head(5), '\n')
print(df['Assay Subcellular Fraction'].dropna().unique(), '\n')

Top five column type counts: 
 Molecule ChEMBL ID    1
Molecule Name         1
Molecule Max Phase    1
Molecular Weight      1
#RO5 Violations       1
dtype: int64 

<StringArray>
['Membrane']
Length: 1, dtype: str 



* No columns have > 1 datatype. Assay Subcellular Fraction contains only the string 'Membrane' and nulls.

#### Missing values and distinct values

In [ ]:
print(f"There are {df.isna().sum()[df.isna().sum() > (0.9 * df.shape[0])].count()} columns with >90% missing values, including {df.isna().sum()[df.isna().sum() == df.shape[0]].count()} with all values missing. \nThere are {df.isna().sum()[df.isna().sum() == 0].count()} columns with no missing values.")

summary = pd.DataFrame()
summary['null_values'] = df.isna().sum()
summary['unique_values'] = df.nunique()
print(f"The max unique values in a column is {summary['unique_values'].max()} for {summary['unique_values'].idxmax()}, which is {df['Molecule ChEMBL ID'].count() - summary['unique_values'].max()} short of the full {df['Molecule ChEMBL ID'].count()} rows in the dataset.")

print("Table of missing and unique values for each column:")
print(f"{summary} \n")


There are 22 columns with >90% missing values, including 5 with all values missing. 
There are 16 columns with no missing values.
The max unique values in a column is 107368 for Molecule ChEMBL ID, which is 6518 short of the full 113886 rows in the dataset.
There are 1 duplicated rows. 

Table of missing and unique values for each column:
                            null_values  unique_values
Molecule ChEMBL ID                    0         107368
Molecule Name                    112271            804
Molecule Max Phase               112630              6
Molecular Weight                     49          19248
#RO5 Violations                    1968              5
AlogP                              1968           1072
Compound Key                          0         107088
Smiles                               52         107346
Standard Type                         0             25
Standard Relation                107696              6
Standard Value                     1223           2657

* Missing values
    * 48 columns
    * Columns 27, 28, 31-33 have 0 non-null values
    * There are 22 columns with >90% missing values, including 5 with all values missing. 
    * There are 16 columns with no missing values.
    * Standard Relation has 107696 null values. These operators are usually used when a rate/potency is above/below the accurate detection limit. But I do see '=' being used, so it could be different reporting by the different assays.
* Distinct values
    * The max unique values in a column is 107368 in Molecule ChEMBL ID, which is 6518 short of the 113886 rows
        * Some compounds have multiple rows
    * Assay Organism has only 1 unique value but also 1770 nulls. The dataset was labelled as 100% Homo Sapiens on the ChEMBL website, so will assume all are Homo Sapiens.

#### Duplicates

In [12]:
print(f"There are {df.duplicated().sum()} duplicated rows. \n")
mol_count = df['Molecule ChEMBL ID'].value_counts()
mask = df['Molecule ChEMBL ID'].value_counts() > 1

print(f"There are {mol_count[mask].shape[0]} compounds with more than 1 row, and they have up to {mol_count[mask].max()} rows (in the case of {mol_count[mask].idxmax()}). \n")
print(df['Molecule ChEMBL ID'].value_counts().head(5))

selection = df[df['Molecule ChEMBL ID'] == 'CHEMBL4518483']

print(f"\nCHEMBL4518483 has been tested in {selection['Assay ChEMBL ID'].nunique()} unique assays (Assay ChEMBL ID), each with up to {selection.groupby('Assay ChEMBL ID')['Molecule ChEMBL ID'].agg('count').max()} rows/values")

There are 1 duplicated rows. 

There are 3811 compounds with more than 1 row, and they have up to 59 rows (in the case of CHEMBL4518483). 

Molecule ChEMBL ID
CHEMBL4518483    59
CHEMBL410972     55
CHEMBL414357     33
CHEMBL4084119    29
CHEMBL5186808    23
Name: count, dtype: int64

CHEMBL4518483 has been tested in 49 unique assays (Assay ChEMBL ID), each with up to 6 rows/values


* Duplicates
    * Only 1 fully duplicated row 
    * There are 3811 compounds with more than 1 row, and they have up to 59 rows (CHEMBL4518483)
    * CHEMBL4518483 has been tested in 49 unique assays, each with up to 6 values

#### Units and measurement types

* How many different standard units are there and what are they?

In [ ]:
grouped_std_units = df.groupby('Standard Units')['Molecule ChEMBL ID'].agg('count')
print(f"\nThere are {grouped_std_units.count()} different units, but nM covers {100  * grouped_std_units['nM']/grouped_std_units.sum():.1f}% of the entries.\n")

std_units_df = pd.DataFrame()
std_units_df['Count'] = grouped_std_units = df.groupby('Standard Units')['Molecule ChEMBL ID'].agg('count')
std_units_df['Percentage'] = round(df['Standard Units'].value_counts(normalize=True) * 100, 1)
print(std_units_df)


There are 17 different units, but nM covers 97.6% of the entries.

                 Count  Percentage
Standard Units                    
%                 2115         1.9
10'8pM               1         0.0
10^-1/s              1         0.0
10^-2/s             11         0.0
10^-3/s              4         0.0
10^-5M               1         0.0
10^-6M               3         0.0
10^2/Ms              1         0.0
10^3(1/Ms)           3         0.0
10^3/M.s             2         0.0
10^3/M/s             2         0.0
10^4/Ms              7         0.0
10^5/M.s             1         0.0
hr                   1         0.0
nM              110342        97.6
pmol/L               8         0.0
s-1                552         0.5


* **Key finding**: Standard units contains a mixture of rates, concentrations and percentages. This will need to be filtered/separated
* There are 17 different units, but nM covers 97.6% of the entries so could reasonably filter to nM only.

What are the 25 unique values for Standard Type?

In [ ]:
print(df['Standard Type'].value_counts())

Standard Type
Potency                  107577
EC50                       2365
%Inhib (Mean)              1032
%Max (Mean)                 558
kon                         552
k_off                       552
IC50                        350
Emax                        300
Activity                    274
Ki                           63
Ratio IC50                   55
Ratio EC50                   53
FC                           49
Kd                           20
Inhibition                   19
Efficacy                     19
Ka                           16
Emin                         10
Kdiss                         9
Ratio                         5
Ke                            4
T1/2                          1
Mean fold stimulation         1
RLU                           1
pEC50                         1
Name: count, dtype: int64


* Standard Type contains 25 unique values. 'Potency' is most common (107577), followed by EC50 (2365). But there are also, %Inhib, %Max, various ratios and rates (kon, koff) etc
* Data will be filtered to Potency as it is by far the most common and will give comparable values.

Filtering to only 'Potency' type measurements and dropping empty columns:

In [40]:
print(f"Dataset shape: {df.shape}")

filtered = df[df['Standard Type'] == 'Potency']
print(f"Filtering Standard Type to Potency type only. New shape: {filtered.shape}")
filtered_2 = filtered.dropna(axis=1, how='all') # Drop null columns
print(f"Dropping {filtered.isna().all().sum()} empty columns. New shape: {filtered_2.shape}")


Dataset shape: (113886, 48)
Filtering Standard Type to Potency type only. New shape: (107577, 48)
Dropping 20 empty columns. New shape: (107577, 28)


Which assays are contained in the data:

In [44]:
print(filtered_2['Assay Description'].value_counts(), '\n')
print(filtered_2['Assay ChEMBL ID'].value_counts(), '\n')

filtered_3 = filtered_2[filtered_2['Assay ChEMBL ID'] == 'CHEMBL2114788']

print(f"Filtering to {filtered_2['Assay ChEMBL ID'].value_counts().idxmax()} gives shape: {filtered_3.shape}")

Assay Description
PubChem BioAssay. qHTS of GLP-1 Receptor Inverse Agonists (Inhibition Mode). (Class of assay: confirmatory)     103901
PubChem BioAssay. qHTS of GLP-1 Receptor Agonists. (Class of assay: confirmatory)                                 3541
PubChem BioAssay. qHTS of GLP-1 Receptor Agonists: Hit Validation.   (Class of assay: confirmatory)                135
Name: count, dtype: int64 

Assay ChEMBL ID
CHEMBL2114788    103901
CHEMBL2114931      3541
CHEMBL3215152       135
Name: count, dtype: int64 

Filtering to CHEMBL2114788 gives shape: (103901, 28)


* Assay Description shows three different assays. Large set of inverse agonists (103901), but also two sets of agonists (3541 & 141). Given the opposite pharmacology, the data was filtered to the inverse agonists only (Assay ChEMBL ID = CHEMBL2114788). New shape (103901, 28).

#### Filtering data:

In [41]:
conclusive_rows = df[(df['Standard Type'] == 'Potency') 
                     & (df['Standard Value'] != 28183.8) 
                     & (df['Comment'] != 'inconclusive')
                     & (df['Assay ChEMBL ID'] == 'CHEMBL2114788')].dropna(axis=1, how='all')

print(f"Potency-only inverse agonist dataset shape:{conclusive_rows.shape}")

Potency-only inverse agonist dataset shape:(21412, 28)
